# Basic Plate Solving

This notebook demonstrates how to use tetra3rs to solve a TESS Full Frame Image, including centroid extraction, iterative distortion calibration, and visualization of matched stars.

In [ ]:
import tetra3rs as t3rs
import numpy as np
import math as m
from astropy.io import fits
import pprint
import time
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook"
from IPython.display import Image, display
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import astropy.units as u

In [ ]:
def add_ellipse(fig, cx, cy, a, b, angle_deg, **kwargs):
    """
    cx, cy   : center
    a, b     : semi-axes (width/2, height/2)
    angle_deg: rotation in degrees (CCW)
    """
    theta = np.linspace(0, 2 * np.pi, 300)
    angle = np.deg2rad(angle_deg)
    
    # Parametric ellipse points
    x = a * np.cos(theta)
    y = b * np.sin(theta)
    
    # Rotate
    x_rot = cx + x * np.cos(angle) - y * np.sin(angle)
    y_rot = cy + x * np.sin(angle) + y * np.cos(angle)
    
    fig.add_trace(go.Scatter(
        x=x_rot, y=y_rot,
        mode='lines',
        fill='toself',
        fillcolor='rgba(255,0,0,0.15)',
        showlegend=False,
        **kwargs
    ))

In [ ]:

solver = t3rs.SolverDatabase.generate_from_hipparcos("../../data/hip2.dat", max_fov_deg=15.0,
                                                     pattern_max_error=0.005,
                                                     lattice_field_oversampling=100,
                                                     patterns_per_lattice_field=100,
                                                     verification_stars_per_fov=1000,
                                                     epoch_proper_motion_year=2018)

In [ ]:
fname = "../../data/tess_same_ccd/sector01_cam1_ccd1.fits"
img: np.ndarray = fits.getdata(fname, ext=1, memmap=False)  # type: ignore
header = fits.getheader(fname, ext=1)

img = img.astype(np.float32)
img = img[:2048, 44:2092]  # 2048 x 2048 science region
print(f"image shape: {img.shape}, dtype: {img.dtype}")

centroids = t3rs.extract_centroids(
    img,
    sigma_threshold=250,
    min_pixels=3,
    max_pixels=10000,
    local_bg_block_size=32,
    max_elongation=50.0,
)
print(f"{len(centroids['centroids'])} centroids extracted")

cen = centroids["centroids"]

# -- WCS ground truth --
# Compare at CRPIX (the WCS reference point / optical center).
# The solver boresight = geometric image center; with order-0 distortion
# terms this differs from CRPIX.  For a fair comparison we project the
# CRPIX pixel through the solver's own model.
wcs = WCS(header)
crpix_x = wcs.wcs.crpix[0] - 1.0  # 0-indexed FITS column
crpix_y = wcs.wcs.crpix[1] - 1.0  # 0-indexed FITS row
sky_crpix = wcs.pixel_to_world(crpix_x, crpix_y)
fits_ra = sky_crpix.ra.deg
fits_dec = sky_crpix.dec.deg
print(f"WCS CRPIX (0-idx): ({crpix_x:.1f}, {crpix_y:.1f})  ->  RA={fits_ra:.4f} Dec={fits_dec:.4f}")

# CRPIX in centered image coords (accounting for col_offset=44 crop)
col_offset = 44
crpix_cx = (crpix_x - col_offset) - img.shape[1] / 2.0
crpix_cy = crpix_y - img.shape[0] / 2.0


# -- Iterative distortion bootstrapping --
n_passes = 5
pass_configs = [
    (0.005, 10, 3, 0.5),
    (0.001, 10, 4, 0.5),
    (0.005, 10, 5, 0.5),
    (0.002, 10, 6, 0.5),
    (0.0001, 10, 6, 0.5),
]

cal = None
fov_est_cammodel = 12
for pass_num in range(1, n_passes + 1):
    mr, ri, dord, fov_err = pass_configs[pass_num - 1]
    print(f"\n{'=' * 60}")
    print(f"Pass {pass_num}: match_radius={mr}, refine_iters={ri}, dist_order={dord}")

    res_cammodel = solver.solve_from_centroids(
        cen,
        fov_estimate_deg=fov_est_cammodel,
        fov_max_error_deg=3,
        image_shape=img.shape,
        match_radius=mr,
        match_threshold=1.0e-5,
        refine_iterations=ri,  
        camera_model=cal,      
        )
    


    if not res_cammodel:
        print(f"Pass {pass_num} FAILED to solve.")
        break

    fov_est_cammodel = res_cammodel.fov_deg


    fits_coord = SkyCoord(ra=fits_ra * u.deg, dec=fits_dec * u.deg)
    # Fair comparison: project CRPIX pixel through solver model
    pred_ra, pred_dec = res_cammodel.pixel_to_world(crpix_cx, crpix_cy)
    pred_coord = SkyCoord(ra=pred_ra * u.deg, dec=pred_dec * u.deg)
    sep_crpix = pred_coord.separation(fits_coord)
    print(f" Cam Model Solver@CRPIX vs WCS@CRPIX:     {sep_crpix.to(u.arcsec):.2f}")


    cal = solver.calibrate_camera(res_cammodel, cen, image_width=img.shape[1], order=dord)

 
    #print(
    #    f"  Distortion fit (order {dord}): "
    #    f"RMSE {dist_result.rmse_before_px:.2f} -> {dist_result.rmse_after_px:.2f} px, "
    #    f"{dist_result.n_inliers} inliers, {dist_result.n_outliers} outliers"
    #)

#print(f"\n{'=' * 60}")
#print(
#    f'Final result: {len(res.matched_catalog_ids)} matches, RMSE {res.rmse_arcsec:.1f}"'
#)


pred_ra, pred_dec = res_cammodel.pixel_to_world(crpix_cx, crpix_cy)
pred_coord = SkyCoord(ra=pred_ra * u.deg, dec=pred_dec * u.deg)
sep = pred_coord.separation(fits_coord)
print(res_cammodel.pixel_to_world(0,0))
#print(f"FITS WCS CRPIX:        RA={fits_ra:.4f}  Dec={fits_dec:.4f}")
##print(f"Solver@CRPIX:          RA={pred_ra:.4f}  Dec={pred_dec:.4f}")
##print(f"Solver boresight:      RA={res.ra_deg:.4f}  Dec={res.dec_deg:.4f}")
#print(f"CRPIX angular sep:     {sep.to(u.arcmin):.4f}  ({sep.to(u.arcsec):.2f})")
#print(f"Boresight angular sep: {sep_boresight.to(u.arcmin):.4f}  ({sep_boresight.to(u.arcsec):.2f})")
print(f"Solver with cammodel@CRPIX: RA={pred_ra:.4f}  Dec={pred_dec:.4f}")
print(f"Solver with cammodel sep:   {sep.to(u.arcmin):.4f}  ({sep.to(u.arcsec):.2f})")
print(f"Solver with cammodel residuals: RMSE {res_cammodel.rmse_arcsec:.2f}")
print(f"Number of matched stars: {len(res_cammodel.matched_catalog_ids)}")




In [ ]:
showplot = True
if showplot:
    fig = px.imshow(img / np.max(img) * 50, color_continuous_scale='gray', zmin=0, zmax=1,
                    binary_string=True, width=800, height=800,
                    title="TESS test image with extracted centroids and matched stars")

    matched_idx_set = set(int(i) for i in res_cammodel.matched_centroids) if res_cammodel else set()

    for ix, c in enumerate(cen):
        cx = c.x + 0.5 * img.shape[1]
        cy = c.y + 0.5 * img.shape[0]
        cov = c.cov
        if cov is not None:
            a     = 6 * np.sqrt(cov[0, 0])
            b     = 6 * np.sqrt(cov[1, 1])
            angle = 0.5 * np.arctan2(2 * cov[1, 0], cov[0, 0] - cov[1, 1]) * 180 / np.pi + 90
            linecolor = 'yellow' if ix in matched_idx_set else 'red'
            linewidth = 1.0 if ix in matched_idx_set else 0.25
            add_ellipse(fig, cx, cy, a, b, angle, line=dict(color=linecolor, width=linewidth))
        else:
            add_ellipse(fig, cx, cy, 3, 3, 0)

    # Draw lines from matched centroids to expected WCS positions
    if res_cammodel:
        # Build WCS once (FITS header covers the full frame; image was cropped to cols 44:2092)
        wcs = WCS(header)
        col_offset = 44  # img col 0 = FITS col 44 (0-indexed)

        line_x, line_y = [], []
        for cid, cidx in zip(res_cammodel.matched_catalog_ids, res_cammodel.matched_centroids):
            star = solver.get_star_by_id(int(cid))
            if star is None:
                continue
            c = cen[int(cidx)]
            act_x = c.x + 0.5 * img.shape[1]
            act_y = c.y + 0.5 * img.shape[0]

            # Expected pixel position from FITS WCS
            sky = SkyCoord(ra=star.ra_deg * u.deg, dec=star.dec_deg * u.deg)
            fits_px, fits_py = wcs.world_to_pixel(sky)
            exp_x = float(fits_px) - col_offset   # convert to img-space
            exp_y = float(fits_py)

            # NaN-separated segments → single efficient trace
            line_x += [act_x, exp_x, None]
            line_y += [act_y, exp_y, None]

        fig.add_trace(go.Scatter(
            x=line_x, y=line_y,
            mode='lines',
            line=dict(color='cyan', width=1),
            showlegend=False,
        ))
    fig.update_xaxes(range=[0, img.shape[1]], scaleanchor='y', scaleratio=1)
    fig.update_yaxes(range=[img.shape[0], 0], scaleanchor='x', scaleratio=1)

    #fig.update_yaxes(autorange="reversed")
    fig.update_coloraxes(showscale=False)
    fig.show()